## Training

In [ ]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.3 MB/s eta 0:00:00


In [ ]:
RUN_COUNTRY = "US"
# THIS MUST MATCH THE ABBREVIATION USED IN THE PATHS
# the training process is around 2 hrs
# should be just within the free limit in Colab

def resolve_country_codes(country_code):
    if country_code == "US":
        return "USA", "US"
    return country_code, country_code

DATA_COUNTRY_CODE, ADAPTER_TAG = resolve_country_codes(RUN_COUNTRY)


In [ ]:
# path to raw files
DATA_DIR = Path("/content/drive/MyDrive/DPO")
# path to train/eval files
DATA_DIR1 = Path("/content/drive/MyDrive/DPO/pre-processing")

RAW_FILE  = DATA_DIR1 / f"{ADAPTER_TAG}_triplets_3594_3.jsonl"
TRAIN_FILE  = DATA_DIR / f"{DATA_COUNTRY_CODE}_3594_train_3.jsonl"

EVAL_FILE  = DATA_DIR / f"{DATA_COUNTRY_CODE}_3594_eval_3.jsonl"

#IN_FILE  = TRAIN_FILE
#OUT_FILE = f"/content/drive/MyDrive/DPO/{DATA_COUNTRY_CODE}_3594_train_with_ref_3.jsonl"

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

DATA_FILE  = f"/content/drive/MyDrive/DPO/{DATA_COUNTRY_CODE}_3594_train_with_ref_3.jsonl"
OUTPUT_DIR = f"/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_3594_{ADAPTER_TAG}_3"

In [ ]:
import os

# Must be set BEFORE importing torch/transformers/trl/accelerate
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["TRANSFORMERS_NO_BF16"] = "1"   # hard-disable bf16 paths in transformers


import torch
torch.set_default_dtype(torch.float16)

print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

CUDA: 12.8
GPU: Tesla T4


In [ ]:
#!pip install trl
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model#, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
# login with hugging face.
# must have an account with them
# occasionally also requests an access token
#!pip install -U "huggingface_hub[cli]"
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import random
from pathlib import Path

#train-test split here. adjust as needed
TRAIN_FRAC = 0.80
SEED = 42


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def split_country_file(raw_path, train_path, eval_path, country, train_frac=0.80, seed=42):
    rows = load_jsonl(raw_path)

    # Add useful metadata for later validation.
    for i, row in enumerate(rows):
        row.setdefault("country", country)
        row.setdefault("item_id", f"{country}_{i:04d}")

    rng = random.Random(seed)
    rng.shuffle(rows)

    n_train = int(len(rows) * train_frac)

    train_rows = rows[:n_train]
    eval_rows = rows[n_train:]

    write_jsonl(train_rows, train_path)
    write_jsonl(eval_rows, eval_path)

    print(f"{country}: {len(rows)} total")
    print(f"  train: {len(train_rows)} -> {train_path}")
    print(f"  eval:  {len(eval_rows)} -> {eval_path}")

    return train_rows, eval_rows


'\nus_train, us_eval = split_country_file(\n    raw_path=US_RAW_FILE,\n    train_path=US_TRAIN_FILE,\n    eval_path=US_EVAL_FILE,\n    country="US",\n    train_frac=TRAIN_FRAC,\n    seed=SEED,\n)\n\nmex_train, mex_eval = split_country_file(\n    raw_path=MEX_RAW_FILE,\n    train_path=MEX_TRAIN_FILE,\n    eval_path=MEX_EVAL_FILE,\n    country="Mexico",\n    train_frac=TRAIN_FRAC,\n    seed=SEED,\n)\n'

In [ ]:
def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )


def format_dataset_example(ex):
    ex["prompt"] = format_prompt_text(tokenizer, ex["prompt"])
    return ex


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Quick VRAM check
free, total = torch.cuda.mem_get_info()
print(f"Free VRAM: {free/1024**3:.2f} GiB / Total: {total/1024**3:.2f} GiB")

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},          # FORCE everything onto GPU 0
    #torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,   # force
)

print("Loaded OK")



Free VRAM: 14.46 GiB / Total: 14.56 GiB


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded OK


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

compute_dtype = torch.float16  # force fp16 to avoid bf16 AMP errors

# Training setup
model.config.use_cache = False

# Manual fallback if needed:
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

##############

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_dataset("json", data_files={"train": DATA_FILE})
train_dataset = ds["train"]

train_dataset = train_dataset.map(format_dataset_example)

##################

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_cfg)

for name, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float16)

# Also cast any floating buffers (rarely needed, but safe)
for name, b in model.named_buffers():
    if b.is_floating_point():
        try:
            model._buffers[name] = b.to(torch.float16)
        except Exception:
            pass

# Sanity check: should now be float16
trainable = next(p for p in model.parameters() if p.requires_grad)
print("Trainable param dtype:", trainable.dtype)

model.print_trainable_parameters()

dpo_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=0.1,
    max_length=768,             # lower this if VRAM is tight
    truncation_mode="keep_end",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    logging_steps=10,
    save_strategy="steps",
    save_steps=5,              # save model weights every training iteration/optimizer step
    save_total_limit=2,      # keep every checkpoint instead of deleting older ones
    report_to="none",

    fp16=True,
    bf16=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,

)

def force_trainable_fp32(m):
    for n, p in m.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)

force_trainable_fp32(trainer.model)

trainable = next(p for p in trainer.model.parameters() if p.requires_grad)
print("Trainable dtype after trainer init:", trainable.dtype)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2875 [00:00<?, ? examples/s]

/tmp/ipykernel_722/2231481664.py:67: FutureWarning: The `'keep_end'` truncation mode is deprecated and will be removed in v2.0.0. Use `truncation_mode='keep_start'` (the default) instead.
  dpo_args = DPOConfig(
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainable param dtype: torch.float16
trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


Adding EOS to train dataset:   0%|          | 0/2875 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2875 [00:00<?, ? examples/s]

Trainable dtype after trainer init: torch.float32


In [ ]:
trainable = next(p for p in model.parameters() if p.requires_grad)
print("Trainable param dtype:", trainable.dtype)
print("Model param dtype:", next(p for p in model.parameters() if p.requires_grad).dtype)
# After trainer is created:
print("Accelerate mixed_precision:", trainer.accelerator.mixed_precision)

bf16_names = [n for n,p in model.named_parameters() if p.requires_grad and p.dtype == torch.bfloat16]
print("BF16 trainable params:", bf16_names[:20], "count:", len(bf16_names))

Trainable param dtype: torch.float32
Model param dtype: torch.float32
Accelerate mixed_precision: fp16
BF16 trainable params: [] count: 0


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,0.654687
20,0.590625
30,0.566016
40,0.585547
50,0.527734
60,0.495312
70,0.599609
80,0.533203
90,0.500391
100,0.450781


Step,Training Loss
10,0.654687
20,0.590625
30,0.566016
40,0.585547
50,0.527734
60,0.495312
70,0.599609
80,0.533203
90,0.500391
100,0.450781


TrainOutput(global_step=180, training_loss=0.5199435763888889, metrics={'train_runtime': 6656.1496, 'train_samples_per_second': 0.432, 'train_steps_per_second': 0.027, 'total_flos': 6.24979353355223e+16, 'train_loss': 0.5199435763888889, 'epoch': 1.0})

In [ ]:
# Save adapter-only
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter-only model to:", OUTPUT_DIR)

Saved adapter-only model to: /content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_3594_US_3
